# Fine-tune XTTS với Dataset ngochuyen_voice

Notebook này hướng dẫn chi tiết cách fine-tune mô hình XTTS với dataset `pnnbao-ump/ngochuyen_voice` từ Hugging Face.

## Thông tin dataset:
- **Tên**: pnnbao-ump/ngochuyen_voice
- **Số mẫu**: ~7,540 audio files
- **Kích thước**: ~3.27GB
- **Ngôn ngữ**: Tiếng Việt
- **Định dạng**: Parquet (audio + text)

## Bước 1: Cài đặt dependencies

Đảm bảo đã cài đặt thư viện `datasets` từ Hugging Face:

In [22]:
!source cuda_venv

In [2]:
# Kiểm tra thư viện datasets đã được cài đặt
try:
    import datasets
    print(f"✓ datasets version: {datasets.__version__}")
except ImportError:
    print("✗ Chưa cài đặt thư viện datasets")
    print("Chạy: uv add datasets")

/home/kourain/truyencv/TTS/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ datasets version: 4.4.2


## Bước 2: Tải và xử lý dataset

Sử dụng script `prepare_ngochuyen_dataset.py` để tải dataset từ Hugging Face:

### Preview mode (xem trước 10 mẫu):

In [ ]:
!python prepare_ngochuyen_dataset.py --preview

### Tải toàn bộ dataset (7,540 mẫu):

**Lưu ý**: Quá trình này sẽ mất khoảng 10-20 phút tuỳ vào tốc độ mạng và CPU.

In [ ]:
# Tải toàn bộ dataset
!python prepare_ngochuyen_dataset.py --output-dir data/ngochuyen_voice --train-split 0.95

## Bước 3: Kiểm tra dataset đã tải

Xem thông tin manifest và dataset:

In [9]:
import pandas as pd
from pathlib import Path
import soundfile as sf

DATASET_ROOT = Path("data/ngochuyen_voice")
TRAIN_MANIFEST = DATASET_ROOT / "metadata_train.csv"
VAL_MANIFEST = DATASET_ROOT / "metadata_val.csv"

# Đọc train manifest
if TRAIN_MANIFEST.exists():
    train_df = pd.read_csv(TRAIN_MANIFEST, sep="|")
    print(f"Train samples: {len(train_df)}")
    print("\nMẫu train đầu tiên:")
    display(train_df.head())
else:
    print("Chưa có train manifest. Chạy script tải dataset trước.")

# Đọc val manifest
if VAL_MANIFEST.exists():
    val_df = pd.read_csv(VAL_MANIFEST, sep="|")
    print(f"\nVal samples: {len(val_df)}")
    display(val_df.head())
else:
    print("Chưa có val manifest")

Train samples: 7163

Mẫu train đầu tiên:


,audio_file,text,speaker_name
0,wavs/audio_00000.wav,"Ngày 23/4/2024, Cục Cảnh sát giao thông đưa ra...",speaker_001
1,wavs/audio_00001.wav,"Theo đó, Lễ kỷ niệm 70 năm Chiến thắng Điện Bi...",speaker_001
2,wavs/audio_00002.wav,Từ Sơn La: Sơn La -> tỉnh lộ 106 -> xã Mường K...,speaker_001
3,wavs/audio_00003.wav,Trong thời gian diễn ra Lễ kỷ niệm 70 năm Chiế...,speaker_001
4,wavs/audio_00004.wav,Tài xế cần kiểm tra yếu tố an toàn của phương ...,speaker_001



Val samples: 377


,audio_file,text,speaker_name
0,wavs/audio_07163.wav,"Phần Trắc nghiệm, tổng số gồm 80 câu hỏi, tổng...",speaker_001
1,wavs/audio_07164.wav,"11. Câu hỏi ở mức độ biết, thông hiểu chiếm 30...",speaker_001
2,wavs/audio_07165.wav,Phần Trắc nghiệm bắt buộc: Môn Toán học có 35 ...,speaker_001
3,wavs/audio_07166.wav,"Phần Trắc nghiệm tự chọn: 15 câu hỏi, tổng điể...",speaker_001
4,wavs/audio_07167.wav,2. Số lượng tuyển chọn: (1) Vị trí việc làm Qu...,speaker_001


In [10]:
# Kiểm tra một file audio mẫu
if TRAIN_MANIFEST.exists() and len(train_df) > 0:
    sample_audio = DATASET_ROOT / train_df["audio_file"].iloc[0]
    if sample_audio.exists():
        info = sf.info(sample_audio)
        print(f"Audio file: {sample_audio.name}")
        print(f"Sample rate: {info.samplerate} Hz")
        print(f"Duration: {info.duration:.2f}s")
        print(f"Channels: {info.channels}")
        print(f"Text: {train_df['text'].iloc[0]}")
    else:
        print(f"Không tìm thấy file: {sample_audio}")

Audio file: audio_00000.wav
Sample rate: 24000 Hz
Duration: 8.67s
Channels: 1
Text: Ngày 23/4/2024, Cục Cảnh sát giao thông đưa ra khuyến cáo về giao thông trong khoảng thời gian diễn ra Lễ kỷ niệm 70 năm Chiến thắng Điện Biên Phủ.


## Bước 4: Cấu hình training

Tạo config file cho training với dataset mới:

In [ ]:
import json
from copy import deepcopy
from datetime import datetime

PROJECT_ROOT = Path.cwd()
CONFIG_TEMPLATE = PROJECT_ROOT / "models" / "config.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "xtts_ngochuyen_ft"
CUSTOM_CONFIG = OUTPUT_DIR / "config.ngochuyen.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Đọc config template
with CONFIG_TEMPLATE.open("r", encoding="utf-8") as f:
    config = json.load(f)

# Cập nhật config cho dataset ngochuyen
config["run_name"] = "xtts_ngochuyen_finetune"
config["run_description"] = f"XTTS fine-tuning with ngochuyen_voice dataset ({datetime.utcnow().isoformat()}Z)"
config["output_path"] = str(OUTPUT_DIR)
config["epochs"] = 10  # Có thể điều chỉnh
config["batch_size"] = 3
config["eval_batch_size"] = 3
config["lr"] = 5e-6

# Cập nhật dataset config
if not config.get("datasets"):
    config["datasets"] = [{}]

dataset_cfg = config["datasets"][0]
dataset_cfg["formatter"] = "coqui"
dataset_cfg["dataset_name"] = "ngochuyen_voice"
dataset_cfg["path"] = str(DATASET_ROOT)
dataset_cfg["meta_file_train"] = "metadata_train.csv"
dataset_cfg["meta_file_val"] = "metadata_val.csv"
dataset_cfg["language"] = "vi"
dataset_cfg["phonemizer"] = ""
dataset_cfg["ignored_speakers"] = None
dataset_cfg["meta_file_attn_mask"] = ""

config["datasets"][0] = dataset_cfg

# Lưu config
with CUSTOM_CONFIG.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"✓ Đã tạo config: {CUSTOM_CONFIG}")
print(f"  Dataset path: {dataset_cfg['path']}")
print(f"  Train manifest: {dataset_cfg['meta_file_train']}")
print(f"  Val manifest: {dataset_cfg['meta_file_val']}")
print(f"  Epochs: {config['epochs']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Learning rate: {config['lr']}")

✓ Đã tạo config: /home/kourain/truyencv/TTS/outputs/xtts_ngochuyen_ft/config.ngochuyen.json
  Dataset path: data/ngochuyen_voice
  Train manifest: metadata_train.csv
  Val manifest: metadata_val.csv
  Epochs: 10
  Batch size: 3
  Learning rate: 5e-06


## Bước 5: Kiểm tra dataset loader

Đảm bảo dataset có thể được load đúng cách bởi TTS:

In [12]:
from TTS.config import load_config
from TTS.tts.datasets import load_tts_samples

cfg = load_config(str(CUSTOM_CONFIG))
train_samples, eval_samples = load_tts_samples(
    cfg.datasets,
    eval_split=cfg.run_eval,
    eval_split_max_size=cfg.eval_split_max_size,
    eval_split_size=cfg.eval_split_size,
)

print(f"✓ Train samples: {len(train_samples)}")
print(f"✓ Eval samples: {len(eval_samples) if eval_samples else 0}")

if train_samples:
    sample = train_samples[0]
    print(f"\nMẫu đầu tiên:")
    print(f"  Audio: {sample['audio_file']}")
    print(f"  Text: {sample['text'][:100]}...")
    print(f"  Speaker: {sample.get('speaker_name', 'N/A')}")

 > Missing column in line 5787 -> wavs/audio_05786.wav|"| Trong năm 2025 và các năm tiếp theo, công tác phòng chống tham nhũng sẽ tiếp tục được Bộ Công an đẩy mạnh quyết liệt trên tinh thần cuộc chiến này không bao giờ “chùng xuống”, không có “ngoại lệ”, không có “vùng cấm”, “bất kể người đó là ai” và “đấu tranh phòng chống tham nhũng không cản trở phát triển kinh tế”."|speaker_001
 | > Found 7163 files in /home/kourain/truyencv/TTS/data/ngochuyen_voice
✓ Train samples: 7163
✓ Eval samples: 377

Mẫu đầu tiên:
  Audio: data/ngochuyen_voice/wavs/audio_00000.wav
  Text: Ngày 23/4/2024, Cục Cảnh sát giao thông đưa ra khuyến cáo về giao thông trong khoảng thời gian diễn ...
  Speaker: speaker_001


## Bước 6: Bắt đầu training

### Option 1: Chạy training trong notebook

In [25]:
PROJECT_ROOT

PosixPath('/home/kourain/truyencv/TTS')

In [29]:
import sys
import os
import subprocess
train_script = f"{PROJECT_ROOT.as_posix()}/TTS/bin/train_tts.py"
SRC_TTS = f"{PROJECT_ROOT.as_posix()}/TTS"
CP_TO = f"{PROJECT_ROOT.as_posix()}/.venv/lib/python3.11/site-packages/"
os.system(f"cp -r {SRC_TTS} {CP_TO}")

cmd = [
    sys.executable,
    str(train_script),
    "--config_path",
    str(CUSTOM_CONFIG),
]

print("Lệnh training:")
print(" ".join(str(c) for c in cmd))

# Đặt True để bắt đầu training
START_TRAINING = True

if START_TRAINING:
    process = subprocess.run(cmd, cwd=PROJECT_ROOT)
    print(f"Training kết thúc với exit code: {process.returncode}")
else:
    print("\nĐặt START_TRAINING=True để bắt đầu training")

Lệnh training:
/home/kourain/truyencv/TTS/.venv/bin/python /home/kourain/truyencv/TTS/TTS/bin/train_tts.py --config_path /home/kourain/truyencv/TTS/outputs/xtts_ngochuyen_ft/config.ngochuyen.json


/home/kourain/truyencv/TTS/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


 > Missing column in line 5787 -> wavs/audio_05786.wav|"| Trong năm 2025 và các năm tiếp theo, công tác phòng chống tham nhũng sẽ tiếp tục được Bộ Công an đẩy mạnh quyết liệt trên tinh thần cuộc chiến này không bao giờ “chùng xuống”, không có “ngoại lệ”, không có “vùng cấm”, “bất kể người đó là ai” và “đấu tranh phòng chống tham nhũng không cản trở phát triển kinh tế”."|speaker_001
 | > Found 7163 files in /home/kourain/truyencv/TTS/data/ngochuyen_voice
 > Using model: xtts


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 16
 | > Num. of Torch Threads: 8
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/kourain/truyencv/TTS/outputs/xtts_ngochuyen_ft/xtts_ngochuyen_finetune-January-10-2026_10+59AM-6991445
Traceback (most recent call last):
  File "/home/kourain/truyencv/TTS/TTS/bin/train_tts.py", line 71, in <module>
    main()
  File "/home/kourain/truyencv/TTS/TTS/bin/train_tts.py", line 58, in main
    trainer = Trainer(
              ^^^^^^^^
  File "/home/kourain/truyencv/TTS/.venv/lib/python3.11/site-packages/trainer/trainer.py", line 506, in __init__
    self.criterion = self.get_criterion(self.model)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kourain/truyencv/TTS/.venv/lib/pyt

Training kết thúc với exit code: 1


### Option 2: Chạy training từ terminal

Mở terminal và chạy lệnh sau:

```bash
cd /home/kourain/truyencv/TTS
source .venv/bin/activate
python TTS/bin/train_tts.py --config_path outputs/xtts_ngochuyen_ft/config.ngochuyen.json
```

## Bước 7: Theo dõi training với TensorBoard

In [ ]:
# Khởi động TensorBoard
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

## Bước 8: Test inference với model đã train

Sau khi training hoàn tất, test mô hình:

In [ ]:
import torch
from TTS.api import TTS as CoquiTTS

# Tìm checkpoint tốt nhất
checkpoint_candidates = sorted(OUTPUT_DIR.glob("best_model*.pth"))
if not checkpoint_candidates:
    checkpoint_candidates = sorted(OUTPUT_DIR.glob("checkpoint_*model.pth"))

if checkpoint_candidates:
    checkpoint_path = checkpoint_candidates[-1]
    print(f"Sử dụng checkpoint: {checkpoint_path}")
    
    # Load model
    tts = CoquiTTS(
        model_path=str(checkpoint_path),
        config_path=str(CUSTOM_CONFIG),
        progress_bar=False,
        gpu=torch.cuda.is_available(),
    )
    
    # Test inference
    test_text = "Xin chào, đây là kiểm thử mô hình XTTS đã fine-tune với dataset ngọc huyền."
    output_wav = OUTPUT_DIR / "test_inference.wav"
    
    # Sử dụng một audio từ dataset làm reference
    if TRAIN_MANIFEST.exists() and len(train_df) > 0:
        ref_audio = DATASET_ROOT / train_df["audio_file"].iloc[0]
        speaker_wav = str(ref_audio) if ref_audio.exists() else None
    else:
        speaker_wav = None
    
    tts.tts_to_file(
        text=test_text,
        speaker_wav=speaker_wav,
        language="vi",
        file_path=str(output_wav),
    )
    
    print(f"✓ Đã tạo audio: {output_wav}")
else:
    print("Chưa có checkpoint. Chạy training trước.")